In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import json


import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn


DATA_PATH = "Data/crsp_ff3_daily_1995_2024.parquet"
SEED = 42
N_STOCKS = 270

features = ["mktrf", "smb", "hml"]
target = "excess_ret"


/Users/nikolauswieland/Documents/masters_thesis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

# Fixed test year (to leave untouched) 
test = df[(df["date"] >= "2024-01-01") & (df["date"] <= "2024-12-31")].copy()

# Pool for training/validation (pre-2024)
pre = df[df["date"] <= "2023-12-31"].copy()

print("Pre-2024 pool:", pre.shape)
print("Test 2024:", test.shape)


Pre-2024 pool: (13064388, 11)
Test 2024: (684286, 11)


In [3]:
SEED_DATES = 123
N_VAL_DATES = 252  # ish 1 trading year

unique_dates = np.sort(pre["date"].unique())
np.random.seed(SEED_DATES)

val_dates = np.random.choice(unique_dates, size=min(N_VAL_DATES, len(unique_dates)), replace=False)
val_dates = pd.to_datetime(val_dates)

# Save them for full universe training later on
pd.Series(pd.to_datetime(val_dates)).sort_values().to_csv("Data/val_dates_hyperparam.csv", index=False, header=["date"])

train = pre[~pre["date"].isin(val_dates)].copy()
val   = pre[ pre["date"].isin(val_dates)].copy()

print("Train:", train.shape)
print("Val  :", val.shape)
print("Test :", test.shape)
print("Unique val dates:", len(val_dates))
print("Val date range:", val["date"].min(), "to", val["date"].max())


Train: (12612270, 11)
Val  : (452118, 11)
Test : (684286, 11)
Unique val dates: 252
Val date range: 1995-01-09 00:00:00 to 2023-10-25 00:00:00


In [4]:
np.random.seed(SEED)
all_permnos = train["permno"].unique()
sample_permnos = np.random.choice(all_permnos, size=N_STOCKS, replace=False)

train_sub = train[train["permno"].isin(sample_permnos)].copy()
val_sub   = val[val["permno"].isin(sample_permnos)].copy()
test_sub  = test[test["permno"].isin(sample_permnos)].copy()

print("Subsample shapes:")
print("Train_sub:", train_sub.shape)
print("Val_sub  :", val_sub.shape)
print("Test_sub :", test_sub.shape)


Subsample shapes:
Train_sub: (1265862, 11)
Val_sub  : (45372, 11)
Test_sub : (68037, 11)


In [5]:
def fit_ff3_ols(stock_train: pd.DataFrame):
    X = sm.add_constant(stock_train[features])
    y = stock_train[target]
    return sm.OLS(y, X).fit()

ff3_models = {}
for permno in sample_permnos:
    stock_train = train_sub[train_sub["permno"] == permno]
    ff3_models[permno] = fit_ff3_ols(stock_train)

ff3_test_preds = []
for permno in sample_permnos:
    model = ff3_models[permno]
    stock_test = test_sub[test_sub["permno"] == permno]
    X_test = sm.add_constant(stock_test[features])
    preds = model.predict(X_test)

    tmp = stock_test[["date", "permno", target]].copy()
    tmp["ff3_pred"] = preds.values
    ff3_test_preds.append(tmp)

ff3_test_results = pd.concat(ff3_test_preds, ignore_index=True)
mse_ff3 = np.mean((ff3_test_results[target] - ff3_test_results["ff3_pred"])**2)

print("FF3 OOS MSE:", mse_ff3)


FF3 OOS MSE: 0.0024984165236791927


In [6]:
class ShallowNN(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
def train_nn(X_train, y_train, X_val, y_val, y_scaler, hidden_dim=16, lr=1e-3, weight_decay=1e-4, epochs=200):
    input_dim = int(X_train.shape[1])
    hidden_dim = int(hidden_dim)
    model = ShallowNN(input_dim=input_dim, hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_train_t)
        loss = loss_fn(preds, y_train_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_preds_scaled = model(X_val_t).numpy().flatten()

    val_preds = y_scaler.inverse_transform(val_preds_scaled.reshape(-1,1)).flatten()
    y_val_orig = y_scaler.inverse_transform(y_val.reshape(-1,1)).flatten()


    val_loss = np.mean((val_preds - y_val_orig)**2)
    
    return model, val_loss


In [8]:
def train_one_stock_nn(stock_train, stock_val, params):
    # X scaling (train only)
    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(stock_train[features])
    X_val   = x_scaler.transform(stock_val[features])

    # y scaling (train only)
    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(stock_train[[target]]).flatten()
    y_val   = y_scaler.transform(stock_val[[target]]).flatten()

    model, val_loss = train_nn(
        X_train, y_train, X_val, y_val, y_scaler,
        hidden_dim=params["hidden_dim"],
        lr=params["lr"],
        weight_decay=params["weight_decay"],
        epochs=params["epochs"],
    )

    return model, x_scaler, y_scaler, val_loss


In [9]:
param_grid = [
    {"hidden_dim": 4,  "lr": 1e-3, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 8,  "lr": 1e-3, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 16,  "lr": 1e-3, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 32,  "lr": 1e-3, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 64, "lr": 1e-3, "weight_decay": 1e-4, "epochs": 200},

    {"hidden_dim": 4,  "lr": 5e-4, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 8,  "lr": 5e-4, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 16,  "lr": 5e-4, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 32,  "lr": 5e-4, "weight_decay": 1e-4, "epochs": 200},
    {"hidden_dim": 64, "lr": 5e-4, "weight_decay": 1e-4, "epochs": 200},

    {"hidden_dim": 4,  "lr": 1e-3, "weight_decay": 5e-4, "epochs": 200},
    {"hidden_dim": 8,  "lr": 1e-3, "weight_decay": 5e-4, "epochs": 200},
    {"hidden_dim": 16, "lr": 1e-3, "weight_decay": 5e-4, "epochs": 200},
    {"hidden_dim": 32,  "lr": 1e-3, "weight_decay": 5e-4, "epochs": 200},
    {"hidden_dim": 64, "lr": 1e-3, "weight_decay": 5e-4, "epochs": 200},
]


In [ ]:
sweep_results = []

for params in tqdm(param_grid, desc="Hyperparameter sweep"):
    val_losses = []

    for permno in sample_permnos:
        stock_train = train_sub[train_sub["permno"] == permno]
        stock_val   = val_sub[val_sub["permno"] == permno]

        # safety
        if len(stock_train) < 200 or len(stock_val) < 50:
            continue

        _, _, _, val_loss = train_one_stock_nn(stock_train, stock_val, params)
        val_losses.append(val_loss)

    avg_val_loss = float(np.mean(val_losses)) if len(val_losses) > 0 else np.inf

    sweep_results.append({
        **params,
        "n_stocks": len(val_losses),
        "avg_val_loss": avg_val_loss
    })

sweep_df = pd.DataFrame(sweep_results).sort_values("avg_val_loss").reset_index(drop=True)
sweep_df


In [11]:
best_params = sweep_df.loc[0].to_dict()
best_params = {
    "hidden_dim": int(best_params["hidden_dim"]),
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "epochs": int(best_params["epochs"]),
}

print("Best params (normalized):", best_params)
print(type(best_params["epochs"]), type(best_params["hidden_dim"]))


Best params (normalized): {'hidden_dim': 16, 'lr': 0.001, 'weight_decay': 0.0005, 'epochs': 200}
<class 'int'> <class 'int'>


In [12]:
trainval_sub = pd.concat([train_sub, val_sub], ignore_index=True)

def train_one_stock_nn_no_val(stock_trainval, params):
    # X scaling (trainval only)
    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(stock_trainval[features].values)

    # y scaling (trainval only)
    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(stock_trainval[[target]].values).flatten()

    # dummy val = train (reuse train_nn; val_loss ignored)
    model, _ = train_nn(
        X_train, y_train, X_train, y_train,
        y_scaler=y_scaler,   
        hidden_dim=params["hidden_dim"],
        lr=params["lr"],
        weight_decay=params["weight_decay"],
        epochs=params["epochs"],
    )

    return model, x_scaler, y_scaler

final_nn_models = {}
for permno in tqdm(sample_permnos, desc="Train final NN (train+val)"):
    stock_trainval = trainval_sub[trainval_sub["permno"] == permno]

    if len(stock_trainval) < 300:
        continue

    model, x_scaler, y_scaler = train_one_stock_nn_no_val(stock_trainval, best_params)
    final_nn_models[permno] = (model, x_scaler, y_scaler)

print("Final NN models trained:", len(final_nn_models))

Train final NN (train+val): 100%|██████████| 270/270 [00:22<00:00, 12.22it/s]

Final NN models trained: 270


In [13]:
nn_test_predictions = []

for permno in sample_permnos:
    if permno not in final_nn_models:
        continue

    model, x_scaler, y_scaler = final_nn_models[permno]
    stock_test = test_sub[test_sub["permno"] == permno]

    if stock_test.empty:
        continue

    X_test = x_scaler.transform(stock_test[features].values)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_t).numpy().flatten()

    preds = y_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()

    tmp = stock_test[["date", "permno", target]].copy()
    tmp["nn_pred"] = preds
    nn_test_predictions.append(tmp)

nn_test_results = pd.concat(nn_test_predictions, ignore_index=True)

mse_nn_best = np.mean((nn_test_results[target] - nn_test_results["nn_pred"])**2)

print("FF3 OOS MSE:", mse_ff3)
print("NN  OOS MSE (best hyperparams):", mse_nn_best)

FF3 OOS MSE: 0.0024984165236791927
NN  OOS MSE (best hyperparams): 0.0025070604503961134


In [14]:
with open("Data/best_nn_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
